In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

world = [
    ('Afghanistan', 'Asia', 652230, 25500100, 20343000000),
    ('Albania', 'Europe', 28748, 2831741, 12960000000),
    ('Algeria', 'Africa', 2381741, 37100000, 188681000000),
    ('Andorra', 'Europe', 468, 78115, 3712000000),
    ('Angola', 'Africa', 1246700, 20609294, 100990000000)
]

world_schema = ["name", "continent", "area", "population", "gdp"]

world_df = spark.createDataFrame(data=world, schema=world_schema)

# Write an SQL query to report the name, population, and area of the big countries.
# big --  area of at least three million (i.e., 3000000 km2), or it has a population of at least twenty-five million (i.e., 25000000).

big_country = world_df.where((col("population") >= 25000000) | (col('area') >= 3000000))
display(big_country)

name,continent,area,population,gdp
Afghanistan,Asia,652230,25500100,20343000000
Algeria,Africa,2381741,37100000,188681000000


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

customer_data = [
    (1, 'Will', None),
    (2, 'Jane', None),
    (3, 'Alex', 2),
    (4, 'Bill', None),
    (5, 'Zack', 1),
    (6, 'Mark', 2)
]

customer_schema = ['id', 'name', 'refree_id']

customer = spark.createDataFrame(customer_data, customer_schema)

result = customer.filter((col('refree_id') != 2) | col('refree_id').isNull())
display(result)

id,name,refree_id
1,Will,null
2,Jane,null
4,Bill,null
5,Zack,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

customer_data = [(1, 'Joe'), (2, 'Henry'), (3, 'Sam'), (4, 'Max')]

order_data = [(1,3),(2,1)]

customer_schema = ['id', 'name']

order_schema = ['id', 'customer_id']

customers = spark.createDataFrame(customer_data, customer_schema)

orders = spark.createDataFrame(order_data, order_schema)

result = customers.join(orders, customers['id']==orders['customer_id'], 'leftanti')
display(result)

id,name
2,Henry
4,Max


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

empz_data = [(1,8),(2,8),(3,8),(4,7),(5,9),(6,9)]
empz_schema = ['id', 'team_id']

empz = spark.createDataFrame(data=empz_data, schema=empz_schema)

WindowSpec = Window.partitionBy('team_id')

result = empz.withColumn('team_size', count('id').over(WindowSpec)).drop('team_id').orderBy(asc("id"))

display(result)

id,team_size
1,3
2,3
3,3
4,1
5,2
6,2


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

schema = [
    "user_id_sender",
    "user_id_receiver",
    "date",
    "action"
]

data = [
    ("ad4943sdz", "948ksx123d", "2020-01-04", "sent"),
    ("ad4943sdz", "948ksx123d", "2020-01-06", "accepted"),
    ("dfdfxf9483", "9djjjd9283", "2020-01-04", "sent"),
    ("dfdfxf9483", "9djjjd9283", "2020-01-15", "accepted"),
    ("ffdfff4234234", "lpjzjdi4949", "2020-01-06", "sent"),
    ("fffkfld9499", "993lsldidif", "2020-01-06", "sent"),
    ("fffkfld9499", "993lsldidif", "2020-01-10", "accepted"),
    ("fg503kdsdd", "ofp049dkd", "2020-01-04", "sent"),
    ("fg503kdsdd", "ofp049dkd", "2020-01-10", "accepted"),
    ("hh643dfert", "847jfkf203", "2020-01-04", "sent"),
    ("r4gfgf2344", "234ddr4545", "2020-01-06", "sent"),
    ("r4gfgf2344", "234ddr4545", "2020-01-11", "accepted"),
]

fb_friend_requests = spark.createDataFrame(data, schema)
# fb_friend_requests.show()

a = fb_friend_requests.filter(col('action')=="sent")
b = fb_friend_requests.filter(col('action')=="accepted")

result = a.join(b, (a['user_id_sender']==b['user_id_sender']) & (a['user_id_receiver']==b['user_id_receiver']), 'inner').groupBy(b.date).agg(count('*').alias('acceptance_count'))

display(result)

date,acceptance_count
2020-01-06,1
2020-01-15,1
2020-01-10,2
2020-01-11,1


In [0]:
schema = [
    "customer_id",
    "city",
    "account_balance",
    "account_status"
]

data = [
    (101, "Mumbai", 12500.50, "Active"),
    (102, "Pune", 0.00, "Inactive"),
    (103, "Delhi", 45890.75, "Active"),
    (104, "Bengaluru", -250.00, "Overdrawn"),
    (105, "Hyderabad", 8900.00, "Active"),
    (106, "Chennai", 120000.25, "Premium"),
    (107, "Kolkata", 3400.80, "Active"),
    (108, "Ahmedabad", 0.00, "Dormant"),
    (109, "Jaipur", 780.50, "Active"),
    (110, "Lucknow", -1250.75, "Overdrawn"),
    (111, "Nagpur", 24500.00, "Premium"),
    (112, "Indore", 5600.40, "Active"),
    (113, "Surat", 150.00, "Inactive"),
    (114, "Bhopal", 99875.30, "Premium"),
    (115, "Visakhapatnam", 4500.00, "Active"),
    (116, "Nashik", 0.00, "Dormant"),
    (117, "Patna", 32500.90, "Active"),
    (118, "Chandigarh", -500.00, "Overdrawn"),
    (119, "Coimbatore", 17250.60, "Active"),
    (120, "Goa", 68500.00, "Premium")
]

customers = spark.createDataFrame(data, schema)
active_customers = customers.where(col('account_status')!='Inactive')

city_wise = active_customers.groupBy('city').agg(avg('account_balance').alias('avg_balance'), count('customer_id').alias('total_customers'))

result = city_wise.filter((col('avg_balance') > 50000) & (col('total_customers') > 0)).select('city', 'avg_balance', 'total_customers')
display(result)

city,avg_balance,total_customers
Chennai,120000.25,1
Bhopal,99875.3,1
Goa,68500.0,1


In [0]:
sf_schema = [
    "id",
    "created_at",
    "value",
    "purchase_id"
]

sf_data = [
    (1, "2019-01-01", 172692, 43),
    (2, "2019-01-05", 177194, 36),
    (3, "2019-01-09", 109513, 30),
    (4, "2019-01-13", 164911, 30),
    (5, "2019-01-17", 198872, 39),
    (6, "2019-01-21", 184853, 31),
    (7, "2019-01-25", 186817, 26),
    (8, "2019-01-29", 137784, 22),
    (9, "2019-02-02", 140032, 25),
    (10, "2019-02-06", 116948, 43),
    (11, "2019-02-10", 162515, 25),
    (12, "2019-02-14", 114256, 12),
    (13, "2019-02-18", 197465, 48),
    (14, "2019-02-22", 120741, 20),
    (15, "2019-02-26", 100074, 49),
    (16, "2019-03-02", 157548, 19),
    (17, "2019-03-06", 105506, 16),
    (18, "2019-03-10", 189351, 46),
    (19, "2019-03-14", 191231, 29),
    (20, "2019-03-18", 120575, 44),
    (21, "2019-03-22", 151688, 47),
    (22, "2019-03-26", 102327, 18),
    (23, "2019-03-30", 156147, 25),
    (24, "2019-04-03", 192530, 36),
    (25, "2019-04-07", 154765, 42),
    (26, "2019-04-11", 113336, 12),
    (27, "2019-04-15", 129073, 50),
    (28, "2019-04-19", 123477, 21),
    (29, "2019-04-23", 182142, 31),
    (30, "2019-04-27", 116546, 39),
    (31, "2019-05-01", 174748, 26),
    (32, "2019-05-05", 155693, 42),
    (33, "2019-05-09", 103012, 25),
    (34, "2019-05-13", 187960, 33),
    (35, "2019-05-17", 101202, 18),
    (36, "2019-05-21", 112522, 10),
    (37, "2019-05-25", 195969, 37),
    (38, "2019-05-29", 117284, 40),
    (39, "2019-06-02", 112956, 36),
    (40, "2019-06-06", 174157, 29),
    (41, "2019-06-10", 125975, 45),
    (42, "2019-06-14", 110340, 29),
    (43, "2019-06-18", 143066, 31),
    (44, "2019-06-22", 153270, 11),
    (45, "2019-06-26", 139635, 29),
    (46, "2019-06-30", 157071, 35),
    (47, "2019-07-04", 166552, 18),
    (48, "2019-07-08", 197587, 34),
    (49, "2019-07-12", 103958, 37),
    (50, "2019-07-16", 111305, 28),
    (51, "2019-07-20", 190266, 39),
    (52, "2019-07-24", 116060, 44),
    (53, "2019-07-28", 163802, 48),
    (54, "2019-08-01", 188558, 21),
    (55, "2019-08-05", 166528, 49),
    (56, "2019-08-09", 141463, 28),
    (57, "2019-08-13", 120110, 34),
    (58, "2019-08-17", 159368, 12),
    (59, "2019-08-21", 184900, 46),
    (60, "2019-08-25", 190372, 38),
    (61, "2019-08-29", 195877, 15),
    (62, "2019-09-02", 143221, 21),
    (63, "2019-09-06", 160413, 41),
    (64, "2019-09-10", 183919, 43),
    (65, "2019-09-14", 134535, 34),
    (66, "2019-09-18", 188646, 31),
    (67, "2019-09-22", 122706, 27),
    (68, "2019-09-26", 160016, 21),
    (69, "2019-09-30", 186777, 21),
    (70, "2019-10-04", 165442, 50),
    (71, "2019-10-08", 172445, 43),
    (72, "2019-10-12", 167910, 21),
    (73, "2019-10-16", 116646, 35),
    (74, "2019-10-20", 163287, 15),
    (75, "2019-10-24", 187293, 43),
    (76, "2019-10-28", 144823, 11),
    (77, "2019-11-01", 118317, 10),
    (78, "2019-11-05", 166105, 38),
    (79, "2019-11-09", 121128, 11),
    (80, "2019-11-13", 177355, 38),
    (81, "2019-11-17", 176442, 50),
    (82, "2019-11-21", 129837, 10),
    (83, "2019-11-25", 122363, 38),
    (84, "2019-11-29", 125469, 10),
    (85, "2019-12-03", 109657, 20),
    (86, "2019-12-07", 108782, 35),
    (87, "2019-12-11", 149235, 18),
    (88, "2019-12-15", 187243, 36),
    (89, "2019-12-19", 152538, 20),
    (90, "2019-12-23", 178861, 34),
    (91, "2019-12-27", 122675, 30),
    (92, "2019-12-31", 104037, 18)
]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

sf_transactions = spark.createDataFrame(sf_data, sf_schema)
# sf_transactions.show(3)

filtered_sf_transactions = sf_transactions.groupBy(date_format("created_at", "yyyy-MM").alias("ym")).agg(sum('value').alias('total'))

WindowSpec = Window.orderBy(col('ym'))

pre_result = filtered_sf_transactions.withColumn('prev_total', lag('total', 1).over(WindowSpec))
result = pre_result.withColumn('growth', round(((col('total') - col('prev_total')) / col('prev_total'))*100, 2)).drop('prev_total','total')
display(result)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ym,growth
2019-01,null
2019-02,-28.56
2019-03,23.35
2019-04,-13.84
2019-05,13.49
2019-06,-2.78
2019-07,-6.0
2019-08,28.36
2019-09,-4.97
2019-10,-12.68


In [0]:
spotify_worldwide_daily_song_ranking = spark.read.csv("/Volumes/workspace/default/sql/spotify_streams.csv", header=True, inferSchema=True)
# display(spotify_worldwide_daily_song_ranking)
result = spotify_worldwide_daily_song_ranking.groupBy('artist').agg(count(col('artist')).alias('n_occurences')).orderBy(desc('n_occurences'))
display(result)

artist,n_occurences
Kendrick Lamar,9
Ed Sheeran,5
Matoma,2
Petit Biscuit,2
The Chainsmokers,2
Wisin,2
Manuel Turizo,2
Sia,2
Migos,2
Zara Larsson,2


In [0]:
athlete_schema = [
    "id",
    "name",
    "sex",
    "age",
    "height",
    "weight",
    "team",
    "noc",
    "games",
    "year",
    "season",
    "city",
    "sport",
    "event",
    "medal"
]

athlete_data = [
    (5, "Christine Jacoba Aaftink", "F", 25, 185, 82, "Netherlands", "NED", "1992 Winter", 1992, "Winter", "Albertville", "Speed Skating", "Speed Skating Women's 500 metres", None),

    (54601, "Jeong Won-Yong", "M", 20, 178, 74, "South Korea", "KOR", "2012 Summer", 2012, "Summer", "London", "Swimming", "Swimming Men's 400 metres Individual Medley", None),

    (94406, "Michael Fred Phelps II", "M", 31, 193, 91, "United States", "USA", "2016 Summer", 2016, "Summer", "Rio de Janeiro", "Swimming", "Swimming Men's 4 x 200 metres Freestyle Relay", "Gold"),

    (60589, "Eliud Kipchoge", "M", 31, 167, 57, "Kenya", "KEN", "2016 Summer", 2016, "Summer", "Rio de Janeiro", "Athletics", "Athletics Men's Marathon", "Gold"),

    (122556, "Blair Tuke", "M", 27, 181, 78, "New Zealand", "NZL", "2016 Summer", 2016, "Summer", "Rio de Janeiro", "Sailing", "Sailing Men's Skiff", "Gold"),

    (124162, "Greg Van Avermaet", "M", 31, 181, 74, "Belgium", "BEL", "2016 Summer", 2016, "Summer", "Rio de Janeiro", "Cycling", "Cycling Men's Road Race Individual", "Gold"),

    (122321, "Irakli Tsirekidze", "M", 26, 184, 90, "Georgia", "GEO", "2008 Summer", 2008, "Summer", "Beijing", "Judo", "Judo Men's Middleweight", "Gold"),

    (30576, "Alina Alexandra Dumitru", "F", 25, 158, 48, "Romania", "ROU", "2008 Summer", 2008, "Summer", "Beijing", "Judo", "Judo Women's Extra-Lightweight", "Gold"),

    (123132, "Masae Ueno", "F", 25, 160, 70, "Japan", "JPN", "2004 Summer", 2004, "Summer", "Athina", "Judo", "Judo Women's Middleweight", "Gold"),

    (999999, "John Testman", "M", 30, 180, 75, "USA", "USA", "2000 Summer", 2000, "Summer", "Sydney", "Athletics", "Athletics Men's 100 metres", "Gold"),

    (999998, "John Testman", "M", 30, 180, 75, "Canada", "CAN", "2004 Summer", 2004, "Summer", "Athens", "Athletics", "Athletics Men's 200 metres", "Bronze")
]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

olympics_athletes_events = spark.createDataFrame(athlete_data, athlete_schema)
# olympics_athletes_events.show(3)

olympics_athletes = olympics_athletes_events.agg(
    min('age').alias('lowest_age'),
    round(avg('age'), 2).alias('mean_age'),
    max('age').alias('highest_age')
)
display(olympics_athletes)

lowest_age,mean_age,highest_age
20,27.36,31
